In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
#
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/city96/ComfyUI-GGUF
#
!pip install -r /content/ComfyUI/requirements.txt
!pip install -r /content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt
#
%cd /content
#
!curl -Lo /content/ComfyUI/models/clip/wwan21_umt5-xxl-encoder-Q4_K_M.gguf      https://huggingface.co/josemerinom/wan/resolve/main/wan21_umt5-xxl-encoder-Q4_K_M.gguf
!curl -Lo /content/ComfyUI/models/clip_vision/wan21_clip_vision_h.safetensors   https://huggingface.co/josemerinom/wan/resolve/main/wan21_clip_vision_h.safetensors
#
!curl -Lo /content/ComfyUI/models/unet/wan21_i2v-14b-720p-Q4_K_M.gguf           https://huggingface.co/josemerinom/wan/resolve/main/wan21_i2v-14b-720p-Q4_K_M.gguf
!curl -Lo /content/ComfyUI/models/vae/wan21_vae_720.pth                         https://huggingface.co/josemerinom/wan/resolve/main/wan21_vae_720.pth
!curl -Lo /content/ComfyUI/models/loras/wan21_lightx2v_780P.safetensors         https://huggingface.co/josemerinom/wan/resolve/main/wan21_lightx2v_780P.safetensors
#
!curl -Lo /content/ComfyUI/models/loras/wan21_I2V_14B_FusionX_LoRA.safetensors  https://huggingface.co/josemerinom/wan/resolve/main/wan21_I2V_14B_FusionX_LoRA.safetensors
#!curl -Lo /content/ComfyUI/models/loras/wan21_lightx2v_T2V.safetensors         https://huggingface.co/josemerinom/wan/resolve/main/wan21_lightx2v_T2V.safetensors
#

In [ ]:
%cd /content
#
!wget -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
#
%cd /content/ComfyUI
!python main.py --dont-print-server --output-directory "/content/drive/MyDrive/output"